# Ranking & Filtering

Authors:
- Ramon Casas i Luque
- Marina Velasco Barriuso
- Shaoya Zhang

>[!NOTE] Run locally so ../../data/ resolves correctly from the project root.

In this notebook we implement and compare different approaches to measure relevance.

# Part 0: Import libraries and data

#TODO: here explain that we moved the functions we used in part to `utils.py` and modified them slightly so they work as an independent module

In [53]:
import utils
import pandas as pd
import numpy as np

In [54]:
# Open csv with preprocessed data
filename = "../../data/index_info.csv"
index_info, fields = utils.load_index_info(filename)

# Create tf-idf index
index, tf, df, idf = utils.create_index_tfidf(index_info, num_documents=len(index_info), fields=fields)

# Part 1: Implementation of three ranking strategies

## a. TF-IDF + cosine similarity

We already implemented this strategy in part 2. Here we reuse the functions we created to 

In [55]:
# Try one test query
products = pd.read_csv("../../data/fashion_products.csv")
query = utils.TEST_QUERIES[2]
equal_weights = np.array(len(fields)*[1/len(fields)])
ranked_docs = utils.search_tf_idf(query, index, tf, idf, fields, equal_weights)
utils.display_results(query, ranked_docs, products, top=7)

🔎 You searched for 'black sports shoes'

====================== Top 7 results out of 94 ====================== 

1 JAGUAR Football Shoes For Women  (Red, Black) | Vector | Footwear > Men's Footwear | SHOFN3EJZKBG6QUF
   sports shoes

2 Lace Up For Men  (Black) | Bone | Footwear > Men's Footwear | SHOFVXG5XR59AFTH
   Bonexy, a brand well known for its domination over the footwear indust...

3 Lace Up For Men  (Black) | Bone | Footwear > Men's Footwear | SHOFVJX2TP8FRR7P
   Bonexy, a brand well known for its domination over the footwear indust...

4 Loafers For Women  (Black) | Bone | Footwear > Men's Footwear | SHOFX2G6FHVNAHZN
   Bonexy, a brand well known for its domination over the footwear indust...

5 Loafers For Men  (Black) | Bone | Footwear > Men's Footwear | SHOFVHY8MAYUPTZF
   Bonexy, a brand well known for its domination over the footwear indust...

6 RS-7250 Running Shoes For Men  (Black) | Vector | Footwear > Men's Footwear | SHOFN3EJHXHGSGUC
   sports shoes

7 Sneakers For

## b. BM25

$$
RSV_d = \sum_{t \in q} \underbrace{\log \left[ \frac{N}{df_t} \right]}_{\text{IDF}} \cdot \underbrace{\frac{(k_1 + 1)tf_{td}}{k_1((1 - b) + b \times (L_d / L_{ave})) + tf_{td}}}_{\text{tf + doc. length normalization}} \cdot \underbrace{\frac{(k_3 + 1)tf_{tq}}{k_3 + tf_{tq}}}_{\text{Query term}}
$$
One weight for each term in the query. Sum over all term weights.

- $N$: number of docs in collection
- $df_t$: document frequency of term $t$ (how many docs does $t$ appear in?)
- $tf_{td}$: frequency of term $t$ in document $d$ (how many times does $t$ appear in $d$?)
- $L_d$: length of document $d$
- $L_{ave}$: average document length in the whole collection
- $k_1$: tuning parameter controlling importance of document term frequency scaling
- $b$: tuning parameter controlling importance of document length


The **query term is optional**, we can use it for long queries.
- $tf_{tq}$ : frequency of term $t$ in query $q$
- $k_3$ : tuning parameter controlling term frequency scaling of the query

**Definition of "document"**:

For us, a document will be the concatenation of all words in a the following fields of a product: `'title', 'description', 'brand', 'category', 'sub_category', 'seller', 'product_details'`

In [56]:
# doc_lengths[pid] = length of document with pid
# Compute L_ave
index_info["document"] = index_info[fields].apply(
    lambda row: [term for col in row if isinstance(col, list) for term in col],
    axis=1,
) # merged terms of all fields of each PID

doc_lengths = dict(zip(index_info["pid"], index_info["document"].str.len())) # doc length of each PID
average_length = np.mean(index_info["document"].str.len())
print(f"The average document length is {average_length:.2f} terms")

The average document length is 56.47 terms


In [57]:
from collections import Counter, defaultdict

# Create dictionary to store raw term frequencies of each product
# tf_td
tf = defaultdict(dict)
for pid, doc in zip(index_info["pid"], index_info["document"]): # (PID, [merged terms])
    for term, count in Counter(doc).items():
        tf[term][pid] = count

# Usage example
term = 'cotton'
pid = 'TKPFCZ9EA7H5FYZH'
print(f"Term '{term}' shows up in product '{pid}' {tf[term][pid]} times")

Term 'cotton' shows up in product 'TKPFCZ9EA7H5FYZH' 2 times


In [58]:
def bm25_score(pid, query, k1, b, doc_lengths, average_length, tf, idf):
    """_summary_
    
    Args:
        pid: product id
        query: query terms
        k1: hyper-param tuning weight of term frequency
        b: hyper-param tuning weight of document length
        doc_lengths: document length
        average_length: average document length over the collection
        tf: term frequency of the query in the document with pid
        idf: inverse document frequency of term

    Returns:
        BM25 score
    """
    rsv = 0
    for t in utils.build_terms(query):
        term1 = idf[t]
        term2 = ( (k1 + 1)*tf[t][pid] ) / ( k1 * ( (1-b)+b * (doc_lengths[pid]/average_length)) + tf[t][pid] )
        rsv += term1 * term2

    return rsv

def search_bm25(query, k1, b, doc_lengths, average_length, tf, idf):
    """_summary_

    Args:
        query: query terms
        k1: hyper-param tuning weight of term frequency
        b: hyper-param tuning weight of document length
        doc_lengths: document length
        average_length: average document length over the collection
        tf: term frequency of the query in the document with pid
        idf: inverse document frequency of term

    Returns:
        ranked documents sorted by its BM25 scores
    """
    docs = utils.search(query, index=index)

    doc_scores = []
    for pid in docs:
        rsv = bm25_score(pid, query, k1, b, doc_lengths, average_length, tf, idf)
        doc_scores.append((rsv, pid))
    
    doc_scores.sort(reverse=True)
    result_docs = [x[1] for x in doc_scores]
    return result_docs


k1 = 1.2
b = 0.75

ranked_docs = search_bm25(query, k1, b, doc_lengths, average_length, tf, idf)

utils.display_results(query, ranked_docs, df=products, top=7)

🔎 You searched for 'black sports shoes'

====================== Top 7 results out of 94 ====================== 

1 JAGUAR Football Shoes For Women  (Red, Black) | Vector | Footwear > Men's Footwear | SHOFN3EJZKBG6QUF
   sports shoes

2 RS-7250 Running Shoes For Men  (Black) | Vector | Footwear > Men's Footwear | SHOFN3EJHXHGSGUC
   sports shoes

3 Football Shoes For Men  (Green, Black) | Vector | Footwear > Men's Footwear | SHOEWQKVXCQRCSSK
   

4 Breeze Football Shoes For Women  (Green, Black) | Vector | Footwear > Men's Footwear | SHOFFTZNUFSFGGMR
   

5 Basketball Shoes For Men  (Green, Black) | Vector | Footwear > Men's Footwear | SHOEZHDKSGJCSUEJ
   

6 Cricket Shoes For Women  (White, Black) | Vector | Footwear > Men's Footwear | SHOEWQKVYWWZ2CZK
   

7 Cricket Shoes For Women  (White, Black) | Vector | Footwear > Men's Footwear | SHOEWQKV5PHERHXN
   



## c. Our score

In this section we create a custom score to rank our products.

Hint: look into numerical fields

Ideas:
- penalize products without an image --> we can attribute it to product info completeness
- reward products with higher rating
- reward products with discount
- Coverage percentage of query terms in title --> in lab1 we have concluded that title is more important than other fields, so we can apply this, and we can also analyze its pro and cons



Our algorithm: BM25 + PICTURE + RATING + DISCOUNT + TITLE



In [59]:
# #numerical = ['a']
# products.sort_values(by="average_rating")
# products['images'].isna().sum()
# products['images_len'] = products['images'].apply(lambda x: True if len(x) > 10 else False)
# build_terms(products[products['images_len']].iloc[67].title)

In [60]:
# 1. penalize with products withoug an image
import requests
import ast
def is_image_url(url):
    """_summary_

    Args:
        url: image url

    Returns:
        _type_: _description_
    """
    try:
        with requests.get(
            url,
            stream=True,
            timeout=5,
        ) as response:
            content_type = response.headers.get('Content_type', "")
            return (
                response.status_code == 200
                and content_type.lower().startswith('image/')
            )
    except requests.RequestException:
        return False

def has_valid_image(value):
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except(ValueError, SyntaxError):
            return False
        
    return any(
        is_image_url(url.strip())
        for url in value
        if isinstance(url, str) and url.strip()
    )



def has_image_url(value):
    # TODO document
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return False
        
    return isinstance(value, list) and any(
        isinstance(url, str)
        and url.strip().startswith(("https://", "http://"))
        for url in value
    )

# Verification
################ 
# with url verification it takes way too long, 
# so I'd suggest only verifying if it has url
# ############
products["has_image_url"] = products["images"].apply(
    has_image_url
)
print(products["has_image_url"].value_counts())
# Penalize product without image by a factor of 0.8

products["image_factor"] = products["has_image_url"].apply(
    lambda has_image: 1.0 if has_image else 0.8
)


has_image_url
True     27308
False      772
Name: count, dtype: int64


In [61]:
# 2.Rating
def rating_factor(rating, alpha=0.2):
    # TODO document
    if pd.isna(rating):
        return 1.0
    
    normalized_rating = np.clip(float(rating), 0, 5) / 5
    return 1 + alpha * normalized_rating

products["rating_factor"] = products["average_rating"].apply(rating_factor)

In [62]:
# 3. Discount
def discount_factor(discount, beta = 0.1):
    """_summary_

    Args:
        discount: discount rate
        beta: hyper-param controlling the importance of discount
        
    return:
        discount multiplier; missing discounts receives no bonus 
    """
    if pd.isna(discount):
        return 1.0
    discount = np.clip(float(discount), 0, 1)
    return 1 + beta * discount


discount_by_pid = pd.to_numeric(
    products.set_index("pid")["discount"],
    errors="coerce"
)

products['discount_factor'] = products['discount'].apply(discount_factor)


In [63]:
# 4. Title coverage
def title_coverage(query, title_terms, gamma = 0.2):
    """_summary_

    Args:
        query (_type_): _description_
        title_terms (_type_): _description_
        gamma (float, optional): _description_. Defaults to 0.2.

    Returns:
        _type_: _description_
    """
    query_terms = set(utils.build_terms(query))
    
    if not query_terms:
        return 0.0
    
    return 1 + gamma * (len(query_terms.intersection(title_terms)) / len(query_terms))


In [64]:
# verification of our algorithm
# Total factor
quality_factor = (
    products['image_factor'] 
    * products['rating_factor'] 
    * products['discount_factor']
)

products['quality_factor'] = quality_factor

docs = utils.search(query, index)
doc_score = []


for pid in docs:
    bm25 = bm25_score(
        pid, query, k1, b, 
        doc_lengths, average_length,
        tf, idf
    )
    
    # extract quality factor of each product
    quality_factor = products.loc[
        products['pid'] == pid, 'quality_factor'
    ].iloc[0]
    
    # extract title_terms of each product
    title_terms = index_info.loc[
        index_info['pid'] == pid, 'title'
    ].iloc[0]
    title_factor = title_coverage(query, title_terms)
    
    score = bm25 * quality_factor * title_factor
    doc_score.append((score, pid))
    
doc_score.sort(reverse=True)
ranked_docs = [pid for score, pid in doc_score]
utils.display_results(query, ranked_docs, df=products, top=7)
    

🔎 You searched for 'black sports shoes'

====================== Top 7 results out of 94 ====================== 

1 JAGUAR Football Shoes For Women  (Red, Black) | Vector | Footwear > Men's Footwear | SHOFN3EJZKBG6QUF
   sports shoes

2 RS-7250 Running Shoes For Men  (Black) | Vector | Footwear > Men's Footwear | SHOFN3EJHXHGSGUC
   sports shoes

3 Cricket Shoes For Women  (White, Black) | Vector | Footwear > Men's Footwear | SHOEWQKV5PHERHXN
   

4 Sports Shoes For Women  (Black, Yellow) | Vector | Footwear > Men's Footwear | SHOE7TZFE2TCNPH5
   Vector X Is One Of The Established Sports Brands Who Provide Stylish A...

5 Football Shoes For Men  (Green, Black) | Vector | Footwear > Men's Footwear | SHOEWQKVXCQRCSSK
   

6 Running Shoes For Men  (Black, Grey) | Vector | Footwear > Men's Footwear | SHOF3A22BBDQQY8F
   Vector X Is One Of The Established Sports Brands Who Provide Stylish A...

7 Basketball Shoes For Men  (Green, Black) | Vector | Footwear > Men's Footwear | SHOEZHDKSGJCSUEJ

## Comparison of a. b. and c.

Explain how the ranking differs when using TF-IDF and BM25, and think about the pros 
and  cons  of  using  each  of  them.  Regarding your own score, justify the choice of the 
score (pros and cons). HINT: Look into numerical fields that each record has to build your 
score.  


Now, we compare how the ranking differs when using the 3 strategies.

(here we could use our 5 test queries, generate a ranking for each of them using the 3 methods, get top 20 and manually assign similarity and assess using NDGC)

## Part 2: Implementation of word2vec + cosine ranking score

Ref: https://www.geeksforgeeks.org/nlp/word2vec-with-gensim/

In [65]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
import numpy.linalg as la

cols = ['title', 'description']

sentences = []
for col in cols:
    l = index_info[col].to_list()
    sentences.extend(l)

len(sentences)

model = Word2Vec(sentences=sentences, vector_size=256, window=5, min_count=1, workers=4, sg=0)
model.save("word2vec.model")

ModuleNotFoundError: No module named 'gensim'

In [ ]:
# term1 = "jean"
# term2 = "pant"
# try:
#     similarity_12 = model.wv.similarity(term1, term2)
#     print(f"Similarity between term1 and term2: {similarity_12:.4f}")
# except KeyError as e:
#     print(f"KeyError: {e}")

# try:
#     similar_words_1 = model.wv.most_similar(term1, topn=10)
#     print(f"Most similar words to {term1}", similar_words_1)
# except KeyError as e:
#     print(f"KeyError: {e}")

# try:
#     analogy_result = model.wv.most_similar(positive=['red', 'white'], topn=1)
#     print("Analogy result for 'red + white':", analogy_result)
# except KeyError as e:
#     print(f"KeyError: {e}")

Similarity between term1 and term2: 0.1371
Most similar words to jean [('nebraska', 0.8183658123016357), ('close', 0.7450138330459595), ('leaves', 0.6858488917350769), ('sit', 0.6846802830696106), ('stretch', 0.6326818466186523), ('indigo', 0.6297811269760132), ('sleek', 0.6207839846611023), ('frank', 0.6083912253379822), ('built', 0.6071932911872864), ('sprin', 0.6070195436477661)]
Analogy result for 'red + white': [('yellow', 0.9580269455909729)]


In [ ]:
query_terms = utils.build_terms(query)
query_vec = model.wv.get_mean_vector(query_terms)
#TODO: ask whether vectors should be normalized (post_normalize=True)
la.norm(query_vec)

np.float32(0.68608016)

In [ ]:
all(query_vec == (model.wv.get_vector('black', norm=True) +
                  model.wv.get_vector('sport', norm=True) +
                  model.wv.get_vector('shoe', norm=True)) / 3)

True

In [ ]:
def rank_documents(terms, docs, index, idf, tf):
    """
    Perform the ranking of the results of a search based on the tf-idf weights

    :param terms: list of query terms
    :param docs: list of documents, to rank, matching the query
    :param index: inverted index data structure
    :param idf: inverted document frequencies
    :param tf: term frequencies
    :param fields: list of fields where terms can be found (e.g ['title', 'description', ...])
    :param field_weights: list of weights for each field

    Returns:
    Print the list of ranked documents
    """
    
    # Empty query or no doc case
    if not terms or not docs:
        return []
    
    # I'm interested only on the element of the docVector corresponding to the query terms
    # The remaining elements would became 0 when multiplied to the query_vector
    doc_vectors = defaultdict(lambda: [0] * len(terms)) # I call doc_vectors[k] for a nonexistent key k, the key-value pair (k,[0]*len(terms)) will be automatically added to the dictionary
    query_vector = [0] * len(terms)

    # compute the norm for the query tf
    query_terms_count = Counter(terms)  # get the frequency of each term in the query.
    # Example: collections.Counter(["hello","hello","world"]) --> Counter({'hello': 2, 'world': 1})

    query_norm = la.norm(list(query_terms_count.values()))

    # Avoid duplicates in the query vector
    for termIndex, term in enumerate(query_terms_count):  #termIndex is the index of the term in the query
        if term not in index:
            continue

        ## Compute tf*idf(normalize TF as done with documents)
        query_vector[termIndex]=query_terms_count[term]/query_norm * idf[term] # query_vector[0] corresponds to the first term in the query

        # Generate doc_vectors for matching docs
        # index[term][field] contains [doc, positions] records.
        for field, postings in index[term].items():
            for doc, positions in postings:
                if doc in docs:

                    # Sum TF-IDF contributions from all fields
                    doc_vectors[doc][termIndex] += (
                        tf[term][field][doc] * idf[term] * weights[field]
                    )
        
    # Calculate the score of each doc

    # compute the cosine similarity between queyVector and each docVector:
    doc_scores=[[np.dot(curDocVec, query_vector), doc] for doc, curDocVec in doc_vectors.items() ]
    doc_scores.sort(reverse=True)
    result_docs = [x[1] for x in doc_scores]
    return result_docs

# Part 3: Alternative representations of documents

(HINT - what about Doc2vec? Sentence2vec? What are the pros and cons

- Doc2Vec already exists
- search for the pros and cons of each
- we could try applying doc2vec to our corpus and finding differences in ranking (https://radimrehurek.com/gensim/auto_examples/tutorials/run_doc2vec_lee.html)